[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/29_adam_solution.ipynb)

# 🟡 Solution: Implement Adam Optimizer

*Training · Medium*

Reference implementation. Try it yourself in `29_adam.ipynb` first.

---
Implement the **Adam** optimizer.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \qquad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$$\hat{m}_t = \frac{m_t}{1-\beta_1^t} \qquad
  \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

$$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

### Signature
```python
class MyAdam:
    def __init__(self, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def init(self, params): ...                    # -> state
    def update(self, params, grads, state): ...    # -> (new_params, new_state)
```

### Rules
- Do **not** use `optax`
- `params` and `grads` are matching **pytrees** — use `jax.tree.map`, do not
  assume a flat array
- The step counter is **1-based**: the first `update` uses $t=1$
- Must work under `jax.jit`

### What bias correction actually fixes
$m$ and $v$ start at **zero**, so at $t=1$, $m_1 = (1-\beta_1)g_1 = 0.1g_1$ —
a tenth of the true gradient. Without correction the first steps are far too
small, and with $\beta_2 = 0.999$ the second-moment estimate takes thousands of
steps to warm up.

The correction has a sharp observable signature: **with** it, the very first
update has magnitude $\approx \eta$ regardless of the gradient's size, since
$\hat{m}_1/\sqrt{\hat{v}_1} = g/|g| = \pm 1$. That is exactly what the tests
check, and the cleanest way to tell a correct Adam from one missing it.

### Where AdamW differs
AdamW does **not** fold weight decay into the gradient. It applies
$\theta \mathrel{-}= \eta\lambda\theta$ separately, so the decay is not scaled
by $\sqrt{\hat{v}}$. Plain "Adam + L2" decays large-gradient parameters *less*,
which is why AdamW generalises better and is the default for transformers.

### ⚠️ Why init/update instead of step()/zero_grad()
The PyTorch original holds the parameters, reads `p.grad`, and mutates `p` in
place inside `step()`. None of that translates: JAX arrays are immutable, there
is no `.grad` attribute, and a method with hidden mutable state cannot be
`jit`-ed or differentiated through.

So `MyAdam` keeps its name but takes the shape every JAX optimizer has —
`init(params) -> state` and `update(params, grads, state) -> (params, state)`,
the same contract as `optax.adam`. There is also no `zero_grad`: `jax.grad`
returns a fresh gradient tree each call, so gradients never accumulate and the
"forgot to zero" bug cannot be written.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


class MyAdam:
    def __init__(self, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps

    def init(self, params):
        # Zero-initialised moments — which is exactly why bias correction is needed.
        return {
            "m": jax.tree.map(jnp.zeros_like, params),
            "v": jax.tree.map(jnp.zeros_like, params),
            "t": 0,
        }

    def update(self, params, grads, state):
        t = state["t"] + 1                      # 1-based on the first update

        m = jax.tree.map(
            lambda m_, g: self.beta1 * m_ + (1 - self.beta1) * g, state["m"], grads
        )
        v = jax.tree.map(
            lambda v_, g: self.beta2 * v_ + (1 - self.beta2) * g * g, state["v"], grads
        )

        mc = 1 - self.beta1 ** t
        vc = 1 - self.beta2 ** t

        new_params = jax.tree.map(
            lambda p, m_, v_: p - self.lr * (m_ / mc) / (jnp.sqrt(v_ / vc) + self.eps),
            params, m, v,
        )
        return new_params, {"m": m, "v": v, "t": t}

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

opt = MyAdam(lr=0.1)
params = {"w": jnp.array([1.0, -2.0])}
state = opt.init(params)

# Wildly different gradient magnitudes both give a first step of ~lr.
for g in (jnp.array([1e-4, 1e-4]), jnp.array([1e4, 1e4])):
    p, _ = opt.update(params, {"w": g}, state)
    print(f"grad {g[0]:>8.0e} -> step {abs(float(p['w'][0] - 1.0)):.4f}")
print("\nThat invariance is bias correction doing its job.")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("adam")